In [ ]:
# Store Item Demand Forecasting – Store 1 / Item 1
# Modélisation SARIMA + Stock de sécurité

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import os

# Créer le dossier images si nécessaire
os.makedirs("images", exist_ok=True)

# 1. Chargement et préparation

df = pd.read_csv("train.csv")

# Convertir les dates en gérant les erreurs
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])

# Filtrer magasin 1 / article 1
df_filtered = df[(df["store"] == 1) & (df["item"] == 1)].copy()
df_filtered = df_filtered.sort_values("date")


# 2. Analyse exploratoire

plt.figure(figsize=(14,5))
plt.plot(df_filtered["date"], df_filtered["sales"])
plt.title("Ventes quotidiennes - Store 1 / Item 1")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.savefig("images/serie_temporelle.png")
plt.show()

# Saisonnalité hebdomadaire
df_filtered["dayofweek"] = df_filtered["date"].dt.dayofweek
df_filtered.groupby("dayofweek")["sales"].mean().plot(kind="bar")
plt.title("Saisonnalité hebdomadaire moyenne")
plt.savefig("images/saisonnalite.png")
plt.show()


# 3. Modèle SARIMA

split = int(len(df_filtered) * 0.8)
train = df_filtered["sales"][:split]
test = df_filtered["sales"][split:]

model = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,7))
results = model.fit()

pred = results.predict(start=len(train), end=len(train)+len(test)-1)

# 4. Évaluation

mae = mean_absolute_error(test, pred)
rmse = np.sqrt(mean_squared_error(test, pred))
bias = np.mean(test - pred)

print("MAE :", round(mae, 2))
print("RMSE :", round(rmse, 2))
print("Biais :", round(bias, 2))

# Visualisation
plt.figure(figsize=(14,5))
plt.plot(test.index, test.values, label="Réel")
plt.plot(test.index, pred.values, label="Prévu")
plt.legend()
plt.title("Prévision vs Réel")
plt.savefig("images/pred_vs_real.png")
plt.show()

# 5. Stock de sécurité

errors = test - pred
sigma = np.std(errors)
lead_time = 3
Z = 1.65

stock_securite = Z * sigma * np.sqrt(lead_time)
print("Stock de sécurité recommandé :", round(stock_securite))


# 6. Baseline naïve (J-7)

naive_pred = test.shift(7)
valid_index = ~naive_pred.isna()

mae_naive = mean_absolute_error(test[valid_index], naive_pred[valid_index])
rmse_naive = np.sqrt(mean_squared_error(test[valid_index], naive_pred[valid_index]))

print("Baseline Naïve - MAE :", round(mae_naive, 2))
print("Baseline Naïve - RMSE :", round(rmse_naive, 2))
